# Data Retrieval — Polymarket

| Field | Details |
|---|---|
| **Time window** | Jul 2024 – Nov 2024 |
| **Source** | Polymarket — Gamma API (`gamma-api.polymarket.com`) |
| **Method** | REST API — JSON response with daily win probabilities |
| **Content** | Daily market-implied probability of winning per candidate |
| **Volume** | **124** daily observations |
| **Saved to** | `Data/1_Bronze/Polymarket/polymarket_win_probabilities.csv` |

### polymarket_win_probabilities.csv — 124 rows × 3 columns

| Column | Description |
|---|---|
| `date` | Date (YYYY-MM-DD) |
| `Trump (%)` | Market-implied probability of Trump winning (%) |
| `Harris (%)` | Market-implied probability of Harris winning (%) |


In [ ]:
import requests
import json
import pandas as pd
import os


In [ ]:
# Step 1: Fetch event and extract token IDs
slug = "presidential-election-winner-2024"
event = requests.get(f"https://gamma-api.polymarket.com/events?slug={slug}").json()[0]

def get_yes_token(markets, name):
    market = next(m for m in markets if name in m["question"])
    return json.loads(market["clobTokenIds"])[0]  # index 0 = Yes token

trump_token  = get_yes_token(event["markets"], "Trump")
harris_token = get_yes_token(event["markets"], "Harris")


In [ ]:
# Step 2: Fetch daily price history and combine
def get_price_history_full(token_id, label):
    params = {"market": token_id, "interval": "max", "fidelity": 1440}
    resp = requests.get("https://clob.polymarket.com/prices-history", params=params).json()
    df = pd.DataFrame(resp.get("history", []))
    df["datetime_utc"] = pd.to_datetime(df["t"], unit="s")
    df["probability"]  = df["p"].astype(float) * 100
    # 00:00 UTC day D = closing price of day D-1 (US time)
    df["date"] = (df["datetime_utc"] - pd.Timedelta(days=1)).dt.normalize()
    return df[["date", "probability"]].rename(columns={"probability": label})

trump_df  = get_price_history_full(trump_token,  "Trump (%)")
harris_df = get_price_history_full(harris_token, "Harris (%)")

combined_df = pd.merge(trump_df, harris_df, on="date", how="outer").sort_values("date").reset_index(drop=True)
combined_df[["Trump (%)", "Harris (%)"]] = combined_df[["Trump (%)", "Harris (%)"]].round(2)


In [ ]:
# Step 3: Filter to campaign period and save
final = combined_df[combined_df["date"] >= "2024-07-05"].reset_index(drop=True)

output_path = "../Data/1_Bronze/Polymarket/polymarket_win_probabilities.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
final.to_csv(output_path, index=False)
print(f"Saved {len(final)} rows → {output_path}")
final.head()
